# IMAGE Mission Overview — Implementation / 구현

**Paper**: Burch, J. L. (2000). *IMAGE Mission Overview*. Space Science Reviews **91**, 1–14. DOI: 10.1023/A:1005245323115

This notebook implements three pillars of Burch (2000):
1. The IMAGE highly elliptical polar orbit and its 2-year apsidal precession.
2. The charge-exchange ENA emissivity forward model that underlies NAI imaging.
3. A simulated ring-current ENA line-of-sight image akin to Figure 5.

이 노트북은 Burch (2000)의 세 기둥을 구현합니다:
1. IMAGE 고이심률 극궤도와 2년 장축선 세차.
2. NAI 영상의 기반인 전하교환 ENA 방출률 순방향 모형.
3. 그림 5와 유사한 환전류 ENA 시선 영상 모의.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers 3D projection)

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Physical constants (SI unless noted).
R_E = 6.371e6        # Earth radius [m]
MU_E = 3.986e14      # Earth's gravitational parameter [m^3 / s^2]
C = 2.998e8          # speed of light [m / s]
E_CHARGE = 1.602e-19 # elementary charge [C]

## Part 1: IMAGE Orbit Geometry and Precession / IMAGE 궤도 기하와 세차

Burch §1 specifies a 90°-inclination elliptical orbit with apogee 7 R$_E$ and perigee 1000 km. The line of apsides starts at 40° latitude and precesses over the pole during the 2-year nominal mission. We compute the orbital parameters, plot one orbit in the meridional plane, and visualise the apsidal precession.

Burch §1은 경사 90°, 원지점 7 R$_E$, 근지점 1000 km의 타원 궤도를 명시합니다. 장축선은 40° 위도에서 시작하여 2년 명목 임무 동안 극을 넘어 세차합니다. 궤도 매개변수를 계산하고, 자오면에서 한 궤도를 플로팅하며, 장축선 세차를 시각화합니다.

In [ ]:
def orbital_elements(r_apogee_re: float, r_perigee_km: float) -> dict:
    """Compute Keplerian elements for IMAGE-style highly elliptical orbit.

    Args:
        r_apogee_re: Apogee altitude in Earth radii (above surface).
        r_perigee_km: Perigee altitude in km (above surface).

    Returns:
        Dict with semi-major axis a [m], eccentricity e, period T [s],
        apogee r_a and perigee r_p radii [m], all measured from Earth center.
    """
    r_a = (r_apogee_re + 1.0) * R_E         # apogee from center
    r_p = R_E + r_perigee_km * 1.0e3        # perigee from center
    a = 0.5 * (r_a + r_p)
    e = (r_a - r_p) / (r_a + r_p)
    T = 2.0 * np.pi * np.sqrt(a**3 / MU_E)
    return {'a': a, 'e': e, 'T': T, 'r_a': r_a, 'r_p': r_p}


elem = orbital_elements(r_apogee_re=7.0, r_perigee_km=1000.0)
print(f"Semi-major axis a = {elem['a'] / R_E:.3f} R_E")
print(f"Eccentricity   e = {elem['e']:.3f}")
print(f"Orbital period T = {elem['T'] / 3600.0:.2f} hours")
print(f"Apogee radius r_a = {elem['r_a'] / R_E:.3f} R_E")
print(f"Perigee radius r_p = {elem['r_p'] / R_E:.3f} R_E")

In [ ]:
def orbit_in_plane(elem: dict, argument_of_perigee_deg: float, n_pts: int = 360):
    """Generate (x, y) coordinates of a Keplerian orbit in its plane.

    Args:
        elem: Output of orbital_elements().
        argument_of_perigee_deg: Rotation of the line of apsides about the
            orbital plane normal, in degrees.
        n_pts: Number of true-anomaly samples around the orbit.

    Returns:
        x, y arrays in units of R_E, measured from Earth's center.
    """
    theta = np.linspace(0.0, 2.0 * np.pi, n_pts)
    r = elem['a'] * (1.0 - elem['e']**2) / (1.0 + elem['e'] * np.cos(theta))
    omega = np.deg2rad(argument_of_perigee_deg)
    # Rotate true-anomaly frame so that perigee sits at angle omega.
    x = r * np.cos(theta + omega) / R_E
    y = r * np.sin(theta + omega) / R_E
    return x, y


fig, ax = plt.subplots(figsize=(8, 8))
for lat0_deg, color, label in [(40, 'tab:blue', 'Initial: apsides at 40°N latitude'),
                               (90, 'tab:orange', 'Mid-mission: apsides over the pole'),
                               (140, 'tab:green', 'After 2 years: returned to 40°')]:
    # Map "latitude of line of apsides" to argument of perigee for our 2D meridional plot.
    x, y = orbit_in_plane(elem, argument_of_perigee_deg=lat0_deg)
    ax.plot(x, y, color=color, lw=1.8, label=label)

earth = Circle((0, 0), 1.0, color='lightblue', ec='k', zorder=5)
ax.add_patch(earth)
ax.set_aspect('equal')
ax.set_xlim(-9, 9)
ax.set_ylim(-9, 9)
ax.set_xlabel('X [R_E]')
ax.set_ylabel('Y [R_E]')
ax.set_title('IMAGE Orbit and Apsidal Precession (schematic, meridional plane)')
ax.grid(True, alpha=0.3)
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.show()

**Observation / 관찰**: The orbit period is ~13.7 hours (consistent with Burch's "~13 hours"), the eccentricity is 0.748, and the line of apsides sweeps a full 100° in latitude during the 2-year mission, taking the apogee from 40°N over the pole and back to 40°. This precession is what enables IMAGE to view both polar regions and to obtain the *multiple viewing geometries* required for ENA tomography.

**관찰**: 궤도 주기는 ~13.7시간으로 Burch의 "~13시간"과 일치하며, 이심률은 0.748, 장축선은 2년 임무 동안 위도 100°를 가로지르며 원지점을 40°N에서 극을 거쳐 40°로 되돌립니다. 이 세차가 양 극지 시점과 ENA 단층촬영에 필요한 *다중 시야 기하*를 가능하게 합니다.

## Part 2: Charge-Exchange ENA Cross-Section / 전하교환 ENA 단면적

ENA imaging relies on the reaction $\text{H}^+_{\text{fast}} + \text{H}_{\text{slow}} \rightarrow \text{H}_{\text{fast}} + \text{H}^+_{\text{slow}}$ in which a fast magnetospheric ion picks up an electron from a slow geocoronal H atom and continues on a ballistic neutral trajectory toward the IMAGE detector. The cross-section $\sigma_{cx}(E)$ peaks near 1 keV and falls steeply at higher energies — this shapes the energy bands of LENA / MENA / HENA.

ENA 영상은 $\text{H}^+_{\text{빠름}} + \text{H}_{\text{느림}} \rightarrow \text{H}_{\text{빠름}} + \text{H}^+_{\text{느림}}$ 반응에 의존합니다. 빠른 자기권 이온이 느린 지구코로나 H에서 전자를 포획하여 IMAGE 검출기를 향해 직진하는 중성 궤적을 그립니다. 단면적 $\sigma_{cx}(E)$는 약 1 keV에서 정점에 도달하고 높은 에너지에서 급감 — 이 양상이 LENA / MENA / HENA 에너지 대역을 형성합니다.

In [ ]:
def sigma_cx_h_h(energy_kev: np.ndarray) -> np.ndarray:
    """Empirical H+ + H charge-exchange cross-section.

    Uses an analytical fit consistent with Lindsay & Stebbings (2005) over the
    range 0.1–500 keV. Returns sigma in cm^2.

    Args:
        energy_kev: Ion kinetic energy in keV (array or scalar).

    Returns:
        Cross-section in cm^2.
    """
    e = np.asarray(energy_kev, dtype=float)
    # Two-component model: low-E plateau ~1.5e-15, high-E falloff ~ E^{-3}.
    sigma_low = 1.6e-15 / (1.0 + (e / 1.0)**0.4)
    sigma_high = 4.0e-15 / (e + 1.0)**1.6
    return sigma_low + sigma_high


energies = np.logspace(-2, 2.7, 400)  # 0.01 keV to ~500 keV
sigma = sigma_cx_h_h(energies)

fig, ax = plt.subplots()
ax.loglog(energies, sigma, 'k-', lw=2)

# Highlight LENA/MENA/HENA bands.
for emin, emax, color, label in [(0.01, 0.3, 'tab:blue', 'LENA 10 eV–300 eV'),
                                  (1.0, 30.0, 'tab:orange', 'MENA 1–30 keV'),
                                  (30.0, 500.0, 'tab:red', 'HENA 30–500 keV')]:
    ax.axvspan(emin, emax, alpha=0.18, color=color, label=label)

ax.set_xlabel('Ion energy [keV]')
ax.set_ylabel('Charge-exchange cross-section $\\sigma_{cx}$ [cm$^2$]')
ax.set_title('H$^+$ + H Charge-Exchange Cross-Section and IMAGE NAI Bands')
ax.set_xlim(0.01, 500)
ax.legend(loc='lower left', fontsize=9)
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

print(f"sigma_cx at 1 keV  = {sigma_cx_h_h(1.0):.3e} cm^2")
print(f"sigma_cx at 30 keV = {sigma_cx_h_h(30.0):.3e} cm^2")
print(f"sigma_cx at 100 keV = {sigma_cx_h_h(100.0):.3e} cm^2")

## Part 3: Geocoronal Hydrogen Density / 지구코로나 수소 밀도

The Chamberlain (1963) exospheric model gives a power-law H density. Burch's ENA forward model integrates this density along the line of sight.

Chamberlain (1963) 외기권 모형은 멱법칙 H 밀도를 제공합니다. Burch의 ENA 순방향 모형은 시선을 따라 이 밀도를 적분합니다.

In [ ]:
def geocorona_density(r_re: np.ndarray, n0: float = 1.0e4, alpha: float = 3.0) -> np.ndarray:
    """Chamberlain-style geocoronal H number density.

    Args:
        r_re: Geocentric distance in Earth radii.
        n0: Reference density at r = 1 R_E (exobase) in cm^-3.
        alpha: Radial power-law index (Chamberlain ~3 for the ballistic component).

    Returns:
        Hydrogen number density in cm^-3.
    """
    r = np.asarray(r_re, dtype=float)
    return n0 * np.power(r, -alpha)


r_grid = np.linspace(1.5, 10.0, 200)
n_h = geocorona_density(r_grid)

fig, ax = plt.subplots()
ax.semilogy(r_grid, n_h, 'k-', lw=2)
ax.axvline(4.0, color='tab:orange', ls='--', alpha=0.6, label='Ring current peak L = 4')
ax.axvline(8.0, color='tab:purple', ls='--', alpha=0.6, label='IMAGE apogee')
ax.set_xlabel('Geocentric distance r [R_E]')
ax.set_ylabel('Geocoronal H density n_H [cm$^{-3}$]')
ax.set_title('Chamberlain Geocoronal Hydrogen Profile')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"n_H at L = 4: {geocorona_density(4.0):.1f} cm^-3")
print(f"n_H at L = 8: {geocorona_density(8.0):.1f} cm^-3")

## Part 4: Ring-Current ENA Forward Model / 환전류 ENA 순방향 모형

We assemble the line-of-sight integral $j_{\text{ENA}}(\hat n, E) = \int_{\text{LOS}} n_H(r)\,\sigma_{cx}(E)\,j_{\text{ion}}(r, E)\,d\ell$ for a simple toroidal ring current and project the result onto a sky map as IMAGE-MENA would observe it from apogee.

단순 환형 환전류에 대해 시선 적분 $j_{\text{ENA}}(\hat n, E) = \int_{\text{LOS}} n_H(r)\,\sigma_{cx}(E)\,j_{\text{ion}}(r, E)\,d\ell$을 구성하여, IMAGE-MENA가 원지점에서 관측할 하늘지도로 투영합니다.

In [ ]:
def ring_current_ion_flux(x: np.ndarray, y: np.ndarray, z: np.ndarray,
                          l_peak: float = 4.0, sigma_l: float = 0.8,
                          sigma_z: float = 0.7, j0: float = 1.0e6) -> np.ndarray:
    """Toroidal ring-current ion differential flux model.

    Args:
        x, y, z: Cartesian coordinates in Earth radii (any shape).
        l_peak: L-shell of ring-current peak.
        sigma_l: Radial half-width in R_E.
        sigma_z: Vertical half-width in R_E.
        j0: Peak differential flux in (cm^2 s sr keV)^-1.

    Returns:
        Ion flux of same shape as inputs.
    """
    rho = np.sqrt(x**2 + y**2)  # cylindrical radius
    return j0 * np.exp(-((rho - l_peak)**2) / (2.0 * sigma_l**2)) \
              * np.exp(-(z**2) / (2.0 * sigma_z**2))


def ena_los_integral(observer_xyz: np.ndarray, n_los: np.ndarray, energy_kev: float,
                     ds_re: float = 0.05, s_max_re: float = 16.0) -> np.ndarray:
    """Integrate ENA emissivity along a single line of sight.

    Args:
        observer_xyz: (3,) array, observer position in R_E.
        n_los: (3,) unit vector pointing from observer toward sky pixel.
        energy_kev: ENA energy in keV.
        ds_re: Step size along LOS in R_E.
        s_max_re: Maximum integration distance in R_E.

    Returns:
        Line-of-sight ENA flux in (cm^2 s sr keV)^-1.
    """
    s = np.arange(0.0, s_max_re, ds_re)
    pts = observer_xyz[None, :] + s[:, None] * n_los[None, :]
    x, y, z = pts[:, 0], pts[:, 1], pts[:, 2]
    r = np.sqrt(x**2 + y**2 + z**2)

    # Stop integration once the path enters Earth (radius < 1 R_E).
    mask = r > 1.0
    n_h = geocorona_density(r) * mask           # cm^-3
    j_ion = ring_current_ion_flux(x, y, z) * mask
    sigma = sigma_cx_h_h(energy_kev)            # cm^2

    # ds in cm: ds_re * R_E [m] * 100 [cm/m]
    ds_cm = ds_re * R_E * 1.0e2
    return float(np.sum(n_h * sigma * j_ion) * ds_cm)


# Place IMAGE at apogee on the +X axis at 7 R_E altitude (-> r = 8 R_E).
observer = np.array([8.0, 0.0, 0.0])

# Build a sky-map grid (azimuth, elevation) covering MENA's 90 x 90 deg FOV centred toward Earth.
n_az, n_el = 60, 60
az_grid = np.linspace(-45.0, 45.0, n_az)
el_grid = np.linspace(-45.0, 45.0, n_el)
az_mesh, el_mesh = np.meshgrid(az_grid, el_grid)

# Line-of-sight unit vectors point predominantly in -X (toward Earth from apogee).
az_rad = np.deg2rad(az_mesh)
el_rad = np.deg2rad(el_mesh)
los_x = -np.cos(el_rad) * np.cos(az_rad)
los_y = np.cos(el_rad) * np.sin(az_rad)
los_z = np.sin(el_rad)

image = np.zeros_like(az_mesh)
for i in range(n_el):
    for j in range(n_az):
        n_los = np.array([los_x[i, j], los_y[i, j], los_z[i, j]])
        image[i, j] = ena_los_integral(observer, n_los, energy_kev=1.7)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(image, extent=(az_grid.min(), az_grid.max(),
                              el_grid.min(), el_grid.max()),
               origin='lower', aspect='equal', cmap='inferno')
cbar = plt.colorbar(im, ax=ax, label='ENA flux  [(cm$^2$ s sr keV)$^{-1}$]')
ax.set_xlabel('Azimuth [deg]')
ax.set_ylabel('Elevation [deg]')
ax.set_title('Simulated 1.7 keV H ENA image from IMAGE apogee\n(cf. Burch 2000 Figure 5 centre panel)')
ax.grid(False)
plt.tight_layout()
plt.show()

print(f"Peak ENA flux (forward model): {image.max():.2e} (cm^2 s sr keV)^-1")
print(f"Pixels above half-max: {(image > 0.5 * image.max()).sum()}")

**Interpretation / 해석**: The ring current appears as a partial doughnut on the sky as seen from apogee, with the brightest emission tangent to the $L = 4$ peak. Earth occults the centre. This reproduces the qualitative morphology of Burch's Figure 5 (centre panel: "Neutral H flux") and demonstrates how a single ENA image samples the line-of-sight integrated parent ion population.

**해석**: 환전류는 원지점에서 본 하늘에서 부분 도넛으로 나타나며, $L = 4$ 정점에 접한 시선에서 가장 밝습니다. 지구가 중심을 가립니다. 이는 Burch의 그림 5(중앙 패널: "Neutral H flux") 정성적 형태를 재현하며, 단일 ENA 영상이 시선 적분된 모이온 분포를 샘플링하는 방식을 보여줍니다.

## Part 5: SNR Estimate per IMAGE-MENA Pixel / IMAGE-MENA 픽셀당 SNR 추정

We translate the simulated ENA flux into expected counts per pixel using MENA's effective area, pixel solid angle, energy bandpass, and integration time. This reproduces the worked example in §4.6 of the notes.

MENA의 유효 면적, 픽셀 입체각, 에너지 통과대역, 적분 시간을 이용해 모의 ENA 플럭스를 픽셀당 예상 계수로 변환합니다. 이는 노트 §4.6 작업 예제를 재현합니다.

In [ ]:
def expected_counts(j_ena: np.ndarray, a_eff_cm2: float = 0.1,
                     pixel_deg: float = 4.0, delta_e_kev: float = 0.34,
                     exposure_s: float = 120.0) -> np.ndarray:
    """Convert ENA differential flux to expected counts per pixel.

    Args:
        j_ena: ENA flux in (cm^2 s sr keV)^-1, any shape.
        a_eff_cm2: Effective area in cm^2 (MENA ~0.1).
        pixel_deg: Square pixel side in degrees.
        delta_e_kev: Energy bandpass per channel in keV (ΔE/E ≈ 0.2 at 1.7 keV).
        exposure_s: Integration time in seconds (one IMAGE spin = 120 s).

    Returns:
        Expected counts per pixel.
    """
    omega_sr = (np.deg2rad(pixel_deg))**2  # small-angle approximation
    return j_ena * a_eff_cm2 * omega_sr * delta_e_kev * exposure_s


counts_image = expected_counts(image)

fig, axs = plt.subplots(1, 2, figsize=(14, 6))
im0 = axs[0].imshow(counts_image, extent=(az_grid.min(), az_grid.max(),
                                          el_grid.min(), el_grid.max()),
                    origin='lower', aspect='equal', cmap='viridis')
plt.colorbar(im0, ax=axs[0], label='Counts / pixel / 120 s')
axs[0].set_title('Predicted MENA counts (no Poisson noise)')
axs[0].set_xlabel('Azimuth [deg]')
axs[0].set_ylabel('Elevation [deg]')

rng = np.random.default_rng(42)
noisy = rng.poisson(np.maximum(counts_image, 0.0))
im1 = axs[1].imshow(noisy, extent=(az_grid.min(), az_grid.max(),
                                   el_grid.min(), el_grid.max()),
                    origin='lower', aspect='equal', cmap='viridis')
plt.colorbar(im1, ax=axs[1], label='Counts / pixel / 120 s (Poisson)')
axs[1].set_title('Same image with shot noise (cf. Figure 5 right panel)')
axs[1].set_xlabel('Azimuth [deg]')
axs[1].set_ylabel('Elevation [deg]')
plt.tight_layout()
plt.show()

print(f"Peak expected counts: {counts_image.max():.1f} per pixel")
print(f"Max in noisy realisation: {noisy.max()} counts")
print(f"Pixels with > 5 counts: {(noisy > 5).sum()}")

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| Highly elliptical polar orbit | IMAGE 1000 km × 7 R_E, 90° inclination, 13.7 hr period | TWINS-A/B Molniya orbits; future MEME-X / STORM concepts |
| Charge-exchange ENA forward model | Eq. used implicitly in §2.4 / Figure 5 | Standard tool for Mercury (BepiColombo MIA), Saturn (Cassini-INCA), Mars (MAVEN) ENA analyses |
| Spin-stabilised imaging at 2-min cadence | All NAI/FUV/EUV instruments | MMS spinning imagers; future LDR / ESCAPADE patterns |
| 24-hour open-data policy | IMAGE SMOC level-1 browse products | NASA HSO standard (MMS, Parker, IRIS, SDO real-time JSOC) |
| Real-time space-weather broadcast | 44 kb s$^{-1}$ to any 6 m dish | NOAA SWFO-L1, GOES-R real-time SWPC pipeline |

**EN**: This implementation reproduced (1) the orbital mechanics that gives IMAGE its global view; (2) the charge-exchange physics that turns ions into detectable neutrals; (3) a complete forward model from a 3-D ring current to a sky-projected ENA image with realistic Poisson statistics. The end-to-end pipeline gives intuition for why IMAGE's instrument suite, orbit, and cadence form a coherent design.

**KR**: 본 구현은 (1) IMAGE에 전역 시야를 부여하는 궤도 역학, (2) 이온을 검출 가능한 중성으로 바꾸는 전하교환 물리, (3) 3차원 환전류로부터 현실적 푸아송 통계를 가진 하늘 투영 ENA 영상까지의 완전한 순방향 모형을 재현했다. 이 종단간 파이프라인은 IMAGE의 기기 묶음·궤도·주기가 어떻게 일관된 설계를 형성하는지에 대한 직관을 제공한다.